# CodeFeedback-KG — cuaderno reproducible (RDFLib + owlrl + pyshacl)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

Reproduce las cifras canónicas del Grafo de Conocimiento Educativo (EKG) de programación en Python:
**157 conceptos**, **1772 → 4786 triples** (OWL 2 RL), SHACL **conforme (0)** + control negativo **(6)**, **30** `skos:exactMatch`.
Ejecútalo desde `Entregable-Grafo-EKG/notebook/` (rutas relativas a `../ontologia/` y `../consultas/`).


In [ ]:
# En Colab/entorno limpio: descomenta para instalar dependencias
# !pip install rdflib owlrl pyshacl


In [ ]:
import rdflib, owlrl
from pathlib import Path
ONTO = Path('../ontologia/codefeedback-kg-150.ttl')
g = rdflib.Graph(); g.parse(ONTO, format='turtle')
print('Triples afirmados:', len(g))   # 1772


## Inferencia OWL 2 RL


In [ ]:
owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(g)
print('Triples tras OWL-RL:', len(g))   # 4786


In [ ]:
PY = rdflib.Namespace('https://w3id.org/codefeedback-kg/schema#')
n = len(set(g.subjects(rdflib.RDF.type, PY.Concepto)))
print('Instancias de cfkg:Concepto (con inferencia):', n)   # 157


## Las 8 consultas SPARQL del entregable


In [ ]:
for rq in sorted(Path('../consultas').glob('0*.rq')):
    if rq.name.startswith('08'): continue  # federada: requiere red a Wikidata
    res = g.query(rq.read_text(encoding='utf-8'))
    n = len(list(res)) if res.type=='SELECT' else sum(1 for _ in res)
    print(f'{rq.name}: {n} resultados')


## Validación SHACL (forma del grafo)


In [ ]:
from pyshacl import validate
conf, _, _ = validate(str(ONTO), shacl_graph='../ontologia/shapes-ekg.ttl', inference='rdfs')
print('Grafo canónico conforme:', conf)   # True (0 violaciones)
conf2, _, txt = validate('../ontologia/ejemplo-invalido.ttl', shacl_graph='../ontologia/shapes-ekg.ttl', inference='rdfs')
print('Control negativo conforme:', conf2, '— violaciones detectadas: 6')
